### Libraries



In [2]:
!pip install -q \
  datasets \
  transformers \
  huggingface_hub \
  evaluate

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from huggingface_hub import hf_hub_download, login
import pandas as pd
from sklearn.metrics import classification_report
from collections import Counter
import json

### Login to huggingface

In [3]:
login()

### Testing

In [4]:
# Use GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

# Load tokenizer and model with LoRA
base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
lora_repo_id  = "eduhuemar001/tinyllama-german-checkpoints-sentiment"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id)
lora_model = PeftModel.from_pretrained(base_model, lora_repo_id)
lora_model = lora_model.to(device)
lora_model.eval()

cuda


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/4.52M [00:00<?, ?B/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 2048)
        (layers): ModuleList(
          (0-21): 22 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_feat

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

base_model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
model = AutoModelForCausalLM.from_pretrained(base_model_id)
model = model.to(device)
model.eval()

cuda


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rotary_emb): 

In [ ]:
# Download the JSONL file from Hugging Face Hub
json_path = hf_hub_download(
    repo_id="eduhuemar001/tinyllama-german-dataset-sentiment-temp-test",
    filename="test_dataset.json",
    repo_type="dataset"
)

# Load full JSONL
with open(json_path, "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

# Convert to DataFrame and clean
df = pd.DataFrame(data)
df = df[["review_text", "sentiment"]]
df = df.dropna(subset=["review_text", "sentiment"])
df["review_text"] = df["review_text"].astype(str).str.strip()
df["sentiment"] = df["sentiment"].astype(str).str.lower().str.strip()
df = df[df["sentiment"].isin(["positive", "neutral", "negative"])]

# Sample 1000 from each class
df_balanced = pd.concat([
    df[df["sentiment"] == "positive"].sample(n=1000, random_state=42),
    df[df["sentiment"] == "neutral"].sample(n=1000, random_state=42),
    df[df["sentiment"] == "negative"].sample(n=1000, random_state=42)
], ignore_index=True).sample(frac=1, random_state=42)  # Shuffle after combining

# Prompt template
instruction = (
    "### Instruction:\n"
    "Klassifiziere die Stimmung der folgenden Bewertung als 'positive', 'neutral' oder 'negative'.\n\n"
    "### Bewertung:\n"
)
answer_prefix = "\n\n### Antwort:\n"

# Run model inference
model = model.to(device)
model.eval()

true_labels = []
pred_labels = []

for i, row in df_balanced.iterrows():
    text = row["review_text"]
    true_label = row["sentiment"]

    prompt = instruction + text + answer_prefix
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    #output = model.generate(**inputs, max_new_tokens=2)
    output = lora_model.generate(**inputs, max_new_tokens=2)
    decoded = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract prediction
    if "### Antwort:" in decoded:
        answer = decoded.split("### Antwort:")[-1].strip().lower()
        answer = answer.split()[0] if answer.split() else ""
    else:
        answer = decoded.strip().lower()

    # Record results
    true_labels.append(true_label)
    pred_labels.append(answer if answer in ["positive", "neutral", "negative"] else "neutral")

    print(f"\n[{i+1}] Bewertung: {text}")
    print(f"    Wahre Stimmung: {true_label}")
    print(f"    Modellantwort: {answer}")

# Count valid predictions
valid_outputs = ["positive", "neutral", "negative"]
valid_count = sum(1 for pred in pred_labels if pred in valid_outputs)
total_count = len(pred_labels)
valid_percentage = 100 * valid_count / total_count

print(f"\n✅ Gültige Modellantworten: {valid_count} von {total_count} ({valid_percentage:.2f}%)")

# Classification report
print("\nKlassifikationsbericht")
print(classification_report(true_labels, pred_labels, digits=3))


[1] Bewertung: In Berlin mit der S-Bahn zum Event sechs S-Bahn-Linien (S41, S42, S46, S5, S7, S75) ermöglichen eine stressfreie Anreise zu den IFA-Bahnhöfen Messe Süd, Messe Nord/ICC und Westkreuz. Vom Berliner
    Wahre Stimmung: positive
    Modellantwort: neutral

[2] Bewertung: Wohnung, Dachgeschosswohnung in 16341 Panketal zum Kauf | Anlageobjekt vor den Toren von Berlin - vermietete Die angebotene Zweiraumwohnung ist vermietet und befindet sich in einen 1995 gebauten Wohnanlage. Die Wohnung befindet sich in einen sehr gepflegten Zustand und verfügt über ein helles Wohnzimmer mit integrierter Küchenzeile, Schlafzimmer sowie Badez
    Wahre Stimmung: neutral
    Modellantwort: neutral

[3] Bewertung: Re: Deutsche Bahn Konzern "Eigentlich hat die schaffnerjn alles richtig gemacht.... Ihre Tochter hatte kejne ,,gültige"" Fahrkarte und hat eben die Fahrpreis nach erhebung bekommen... Was oder wie alles genau ablief wissen nur die schaffnerjn und ihre Tochter .. Und zum Thema ans Tele